# AT-TPC Latent Space Exploration
---
This notebook explores AT-TPC event embeddings with k-means, PCA, UMAP, t-SNE, and a silhouette score.

In [ ]:
import os
import sys
import numpy as np
sys.path.append('../../')
from clustering import t_SNE_clustering
from clustering import UMAP_embedding
from clustering import k_means_clustering
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
# create a folder for results of global feature exploration

folder_path = './plots/exploration'
if not os.path.exists(folder_path):
  os.makedirs(folder_path, exist_ok=True)

## 1. Load Data
---
Set the feature and label paths, then load the `.npy` files.

By default, labels are used exactly as they appear in the file. Plot legends show numeric values like `0`, `1`, and `2` unless you provide custom names.

**`label_groups`** (optional): combine raw label values into fewer classes. Set to `None` to skip regrouping. Each inner list becomes one class.

**`class_names`** (optional): custom legend names. Leave as `None` for numeric labels. If provided, include one name per unique label after any regrouping.



In [ ]:
# Set file paths
features_path = '../../data/features.npy' # CHANGE THE NAME OF THE DATA FILE AS NEEDED
labels_path = '../../data/master_labels.npy' # CHANGE THE NAME OF THE DATA FILE AS NEEDED

# Optional: regroup raw label values into fewer classes.
# None = use labels from the file as-is (default).
label_groups = None
# label_groups = [
#     [0, 1, 2],
#     [3],
#     [4, 5],
# ]

# Optional display names for plot legends.
# None = use numeric labels like "0", "1", "2".
class_names = None
# class_names = ["Class 0", "Class 1", "Class 2"]

features = np.load(features_path)
print(f"Features loaded successfully! Shape: {features.shape}")

In [ ]:
# Load labels
label_data = np.load(labels_path)

In [ ]:
def apply_label_groups(labels, groups):
    if groups is None:
        return labels
    mapped = labels.copy()
    for new_label, raw_values in enumerate(groups):
        mapped[np.isin(labels, raw_values)] = new_label
    return mapped

labels = np.asarray(label_data).astype(int).ravel()
labels = apply_label_groups(labels, label_groups)

if class_names is not None and len(class_names) != len(np.unique(labels)):
    raise ValueError(
        f"class_names must have {len(np.unique(labels))} entries, got {len(class_names)}"
    )

print("Unique labels:", np.unique(labels))
print("Label counts:", {int(v): int(np.sum(labels == v)) for v in np.unique(labels)})

features = StandardScaler().fit_transform(features)
print(f"Data scaled successfully! {features.shape}")

## 2. K-Means
---
Run k-means on the embeddings, seperating them into k clusters. The clusters can then be compared with the grouped labels, using `class_names` for legend text when it is provided.

In [ ]:
save_dir = './plots/k_means'

features_glob, cluster_labels, cluster_indices = k_means_clustering(
    features,
    labels,
    dimension=2,
    save_dir=save_dir,
    num_samples_to_print=10,
    class_names=class_names
)

## 3. PCA Projection
---
Use PCA as a baseline before UMAP and t-SNE. PCA can project the scaled embeddings into either 2D or 3D by changing `dimension`. It shows the main directions of variation and uses the same labels and optional `class_names` as the other plots.

In [ ]:
from clustering import pca_clustering

save_dir = './plots/pca'

# Use dimension=2 for a 2D plot or dimension=3 for a 3D plot.
pca_results = pca_clustering(
    features,
    labels,
    dimension=2,
    save_dir=save_dir,
    class_names=class_names
)

## 4. Check Cluster Labels
---
Print the true labels for the sampled events in each cluster.

In [ ]:
def true_labels(indices, labels):
    c_labels = []
    for index in indices:
        c_labels.append(labels[index])

    return c_labels

print("True labels verification per cluster:")
for i, cluster in enumerate(cluster_indices):
    print(f"Cluster {i}: {true_labels(cluster, labels)}")

## 5. UMAP Visualization
---
Run UMAP on the embeddings and color the 2D plot by label. The main hyperparameter to change is `neighbors`, which controls how much local structure UMAP preserves. The plot uses numeric labels unless `class_names` is set, saves under `./plots/umap/`, and stores the coordinates in `umap_results`.

In [ ]:
neighbors = 50
plt_colors = ['blue', 'green', 'red']

fig, ax = plt.subplots(figsize=(10, 7))

umap_results = UMAP_embedding(
    features,
    dimension=2,
    ax=ax,
    labels=labels,
    neighbors=neighbors,
    class_names=class_names,
    plt_colors=plt_colors,
    save_dir='./plots'
)

ax.legend()
plt.title(f"UMAP 2D Manifold Embedding (n_neighbors={neighbors})")
plt.savefig('./plots/umap/umap_unified_manifold.png', dpi=200)
plt.show()

## 6. t-SNE Visualization
---
Run t-SNE on the embeddings and color the 2D plot by label. The main hyperparameter to change is `perplexity`, which controls how many nearby points t-SNE considers when building the embedding. The plot uses numeric labels unless `class_names` is set. t-SNE is useful for local structure, but far-apart distances should be interpreted carefully. The plot is saved under `./plots/t-sne/`, and the coordinates are stored in `tsne_results`.

In [ ]:
perplexity = 40
# Add more colors if you have more than 3 labels.
plt_colors = ['blue', 'green', 'red', 'orange', 'purple', 'brown']

fig, ax = plt.subplots(figsize=(10, 7))

tsne_results = t_SNE_clustering(
    features,
    dimension=2,
    ax=ax,
    labels=labels,
    perplexity=perplexity,
    class_names=class_names,
    plt_colors=plt_colors,
    save_dir='./plots'
)

ax.legend()
plt.title(f"t-SNE 2D Manifold Embedding (perplexity={perplexity})")
plt.savefig('./plots/t-sne/tsne_unified_manifold.png', dpi=200)
plt.show()

## 7. Silhouette Score
---
Compute a silhouette score on the embeddings using the labels. This gives a simple score for class separation. Silhouette requires at least two unique labels.

A score near **1.0** means strong separation. A score near **0.0** means weak separation.

In [ ]:
from sklearn.metrics import silhouette_score

unique_label_count = len(np.unique(labels))

if unique_label_count < 2:
    print("Silhouette score skipped: need at least 2 unique labels.")
    print(f"Found {unique_label_count} unique label(s): {np.unique(labels)}")
else:
    sample_size = min(len(features), 2000)
    sil_score = silhouette_score(features, labels, sample_size=sample_size, random_state=42)

    print("=" * 50)
    print(f"Silhouette Score (scaled feature space): {sil_score:.4f}")
    print("=" * 50)

    if sil_score >= 0.7:
        print("Interpretation: Strong, well-separated cluster structure")
    elif sil_score >= 0.5:
        print("Interpretation: Reasonable cluster structure")
    elif sil_score >= 0.25:
        print("Interpretation: Weak cluster structure — representations may need improvement")
    else:
        print("Interpretation: No meaningful separation — contrastive objective needs tuning")